# Problem B — Incident Category Classification: Visualizations

Replicates and extends all charts from the Streamlit **Problem B** page.

| Section | Charts |
|---------|--------|
| EDA | Label distribution, co-occurrence matrix, prevalence over time, primary category pie |
| Model | Per-label thresholds, tier architecture comparison |
| Label Statistics | Multi-label count distribution, category correlation heatmap |

**Pre-req:** Run `scripts/run_phase3.py` to generate `models/category_tfidf_baseline.joblib`.

In [1]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [2]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

MODELS    = Path("models")
PROCESSED = Path("data/processed")

LABEL_COLORS = {
    "ATC_Communication":  "#3498db",
    "Airspace_Navigation": "#9b59b6",
    "Environment":         "#2ecc71",
    "Equipment_System":    "#e67e22",
    "Flight_Operations":   "#e74c3c",
}
LABEL_DISPLAY = {
    "ATC_Communication":  "ATC / Communication",
    "Airspace_Navigation": "Airspace / Navigation",
    "Environment":         "Environment / Weather",
    "Equipment_System":    "Equipment / System",
    "Flight_Operations":   "Flight Operations",
}

print("Libraries loaded.")

Libraries loaded.


## 1. Load Data

In [3]:
cats   = pd.read_parquet(PROCESSED / "category_targets.parquet")
splits = pd.read_parquet(PROCESSED / "temporal_splits.parquet")

label_cols = [c for c in cats.columns if c not in ["acn_num_ACN", "primary_category"]]
merged     = splits.merge(cats, on="acn_num_ACN", how="inner")

print(f"Category targets: {cats.shape}")
print(f"Label columns:    {label_cols}")
print(f"Merged with splits: {merged.shape}")
print(f"Split distribution:\n{merged['split'].value_counts().to_string()}")

Category targets: (38655, 7)
Label columns:    ['ATC_Communication', 'Airspace_Navigation', 'Environment', 'Equipment_System', 'Flight_Operations']
Merged with splits: (38655, 10)
Split distribution:
split
train    22854
test     13309
val       2492


## 2. Training Label Distribution

In [4]:
counts = {LABEL_DISPLAY.get(c, c): int(cats[c].sum()) for c in label_cols}

fig = px.bar(
    x=list(counts.keys()),
    y=list(counts.values()),
    color=list(counts.keys()),
    color_discrete_sequence=list(LABEL_COLORS.values()),
    title="Incident Count per Category (multi-label: one report may have multiple)",
    labels={"x": "Category", "y": "Count"},
)
fig.update_layout(height=380, showlegend=False)
for i, (k, v) in enumerate(counts.items()):
    fig.add_annotation(x=k, y=v + 50, text=f"{v:,}", showarrow=False)
fig.show()

print("\nPositive rates:")
for c in label_cols:
    rate = cats[c].mean()
    print(f"  {LABEL_DISPLAY.get(c,c):30s}: {rate:.1%}")


Positive rates:
  ATC / Communication           : 17.1%
  Airspace / Navigation         : 5.9%
  Environment / Weather         : 22.0%
  Equipment / System            : 42.5%
  Flight Operations             : 74.1%


## 3. Category Co-occurrence Heatmap

In [5]:
co_matrix = cats[label_cols].T.dot(cats[label_cols]).astype(int)
np.fill_diagonal(co_matrix.values, 0)
display_names = [LABEL_DISPLAY.get(c, c) for c in label_cols]

fig = px.imshow(
    co_matrix.values,
    x=display_names,
    y=display_names,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Category Co-occurrence Matrix (off-diagonal: reports with both labels)",
)
fig.update_layout(height=450)
fig.show()

## 4. Category Prevalence Over Time

In [6]:
merged["year"] = merged["year"].astype(int)
time_series    = merged.groupby("year")[label_cols].mean().reset_index()

fig = go.Figure()
for col in label_cols:
    fig.add_trace(go.Scatter(
        x=time_series["year"],
        y=time_series[col] * 100,
        mode="lines+markers",
        name=LABEL_DISPLAY.get(col, col),
        line=dict(color=LABEL_COLORS.get(col, "#999")),
    ))
fig.update_layout(
    title="% of Reports Classified to Each Category by Year",
    xaxis_title="Year",
    yaxis_title="% of Reports",
    height=420,
    hovermode="x unified",
)
fig.show()

## 5. Primary Category Distribution (Pie)

In [7]:
primary_counts = cats["primary_category"].value_counts().reset_index()
primary_counts.columns = ["category", "count"]
primary_counts["display"] = primary_counts["category"].map(LABEL_DISPLAY).fillna(primary_counts["category"])
primary_counts["color"]   = primary_counts["category"].map(LABEL_COLORS).fillna("#999")

fig = px.pie(
    primary_counts,
    names="display",
    values="count",
    color="display",
    color_discrete_sequence=primary_counts["color"].tolist(),
    title="Primary Category Distribution (most prevalent label per report)",
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.update_layout(height=450)
fig.show()

## 6. Multi-Label Count Distribution

How many labels does a single report typically get?

In [8]:
label_count = cats[label_cols].sum(axis=1).value_counts().sort_index()

fig = px.bar(
    x=label_count.index,
    y=label_count.values,
    title="Number of Category Labels per Report",
    labels={"x": "Label Count per Report", "y": "Number of Reports"},
    color=label_count.index,
    color_continuous_scale="Blues",
    text=label_count.values,
)
fig.update_traces(textposition="outside")
fig.update_layout(height=380, showlegend=False)
fig.show()

print(f"Avg labels per report: {cats[label_cols].sum(axis=1).mean():.2f}")
print(f"Unlabeled reports: {(cats[label_cols].sum(axis=1)==0).sum():,}")

Avg labels per report: 1.62
Unlabeled reports: 139


## 7. Model Tier Architecture & Per-Label Thresholds

In [9]:
import joblib

try:
    tfidf_model = joblib.load(MODELS / "category_tfidf_baseline.joblib")
    label_names = tfidf_model["label_names"]
    thresholds  = tfidf_model["thresholds"]

    thresh_df = pd.DataFrame([
        {"Category": LABEL_DISPLAY.get(l, l), "Threshold": thresholds[l]}
        for l in label_names
    ]).set_index("Category")
    print("Per-label decision thresholds (tuned on validation set):")
    print(thresh_df.to_string())

    fig = px.bar(
        x=[LABEL_DISPLAY.get(l, l) for l in label_names],
        y=[thresholds[l] for l in label_names],
        color=[LABEL_COLORS.get(l, "#999") for l in label_names],
        title="Per-Label Decision Thresholds (higher = more conservative)",
        labels={"x": "Category", "y": "Threshold Probability"},
    )
    fig.add_hline(y=0.5, line_dash="dash", line_color="gray", annotation_text="Default 0.5")
    fig.update_layout(height=350, showlegend=False)
    fig.show()

except FileNotFoundError:
    print("TF-IDF model not found — run scripts/run_phase3.py first.")

Per-label decision thresholds (tuned on validation set):
                       Threshold
Category                        
ATC / Communication         0.55
Airspace / Navigation       0.90
Environment / Weather       0.60
Equipment / System          0.45
Flight Operations           0.55


## 8. Category Correlation Matrix (Phi Coefficient)

How correlated are the category labels?

In [10]:
label_data = cats[label_cols].astype(float)
corr       = label_data.corr()
corr.index   = [LABEL_DISPLAY.get(c, c) for c in label_cols]
corr.columns = [LABEL_DISPLAY.get(c, c) for c in label_cols]

fig = px.imshow(
    corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    zmin=-1, zmax=1,
    text_auto=".2f",
    title="Label Correlation Matrix (Pearson)",
)
fig.update_layout(height=450)
fig.show()

## 9. Category Trends by Split (Train / Val / Test)

In [11]:
split_rates = (
    merged.groupby("split")[label_cols]
    .mean()
    .mul(100)
    .rename(columns=LABEL_DISPLAY)
    .reset_index()
)

split_long = split_rates.melt(id_vars="split", var_name="Category", value_name="Prevalence (%)")

fig = px.bar(
    split_long,
    x="split",
    y="Prevalence (%)",
    color="Category",
    barmode="group",
    title="Category Prevalence by Train/Val/Test Split",
    labels={"split": "Split"},
    category_orders={"split": ["train", "val", "test"]},
    color_discrete_sequence=list(LABEL_COLORS.values()),
)
fig.update_layout(height=420)
fig.show()